# NB07 — Multi-Tile Potsdam + Darmstadt ZS

**Part 1 — Potsdam (3 tiles, all 4 methods, mIoU)**
- Tile 6_15 (official val), 5_14, 7_13
- Methods: A ZS-single | B ZS-multi | C ZS-multi+PAMR | D PTSAM+PAMR

**Part 2 — Darmstadt DOP20 (all 4 methods, visual only)**
- 5 randomly sampled 256×256 patches
- 1 full reconstructed tile (stitched from all patches of one scene)
- Uses `cls_hessen.txt` (6 classes incl. railway)
- PTSAM soft_prompts are cross-domain here (trained on Potsdam) — interesting to compare

**Datasets:** sam3-weights · 6isprs · ptsam-soft-prompts · darmstadt-dop20-presliced

## 1 — Environment setup

In [ ]:
import os
!wget -q https://repo.anaconda.com/miniconda/Miniconda3-latest-Linux-x86_64.sh -O /tmp/miniconda_installer.sh
!bash /tmp/miniconda_installer.sh -b -p /tmp/miniconda
os.environ.pop('PYTHONPATH', None)
os.environ['PATH'] = '/tmp/miniconda/bin:' + os.environ['PATH']
!conda tos accept --override-channels --channel https://repo.anaconda.com/pkgs/main
!conda tos accept --override-channels --channel https://repo.anaconda.com/pkgs/r
!conda --version

In [ ]:
!/tmp/miniconda/bin/conda create -n segearth python=3.10 -y

In [ ]:
!conda run -n segearth pip install torch==2.4.0 torchvision==0.19.0 tifffile matplotlib -q

## 2 — Clone repo

In [ ]:
import subprocess, os
from pathlib import Path
REPO = Path('/tmp/SegEarth-OV-3')
if REPO.exists():
    subprocess.run(['git', '-C', str(REPO), 'pull', '--ff-only'], check=True)
else:
    subprocess.run(['git', 'clone', '--depth=1',
                    'https://github.com/HarishDeepak/rg-segearth-ov3', str(REPO)], check=True)
os.chdir(REPO)
!conda run -n segearth pip install -r requirements.txt -q

## 3 — Inference

In [ ]:
%%bash
export MPLBACKEND=Agg
export PYTHONUNBUFFERED=1
source /tmp/miniconda/bin/activate segearth
cd /tmp/SegEarth-OV-3

python - << 'PYEOF'
import sys, torch, torch.nn.functional as F
import numpy as np, tifffile, random
from pathlib import Path
from PIL import Image
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

sys.stdout.reconfigure(line_buffering=True)
random.seed(42)

DEVICE               = 'cuda'
POTSDAM_ROOT         = Path('/kaggle/input/datasets/dummyirl/6isprs')
DARMSTADT_ROOT       = Path('/kaggle/input/datasets/harish77718/darmstadt-dop20-presliced/darmstadt_dop20/images')
POTSDAM_TILES        = ['6_15', '5_14', '7_13']   # 6_15 = official val
N_CLASSES_P          = 6    # Potsdam
N_CLASSES_H          = 6    # Hessen
BG_IDX_P             = 5    # clutter
BG_IDX_H             = 0    # impervious as fallback (no GT clutter class in Hessen)
IGNORE_IDX           = 255
CROP_SIZE            = 1024
STRIDE               = 1024  # fast (36 crops/tile); use 512 for smoother blending
CONFIDENCE_THRESHOLD = 0.1
PROB_THD             = 0.1
PAMR_ITER            = 10
PAMR_DILATIONS       = [1, 2, 4]
N_DARMSTADT_PATCHES  = 5

# ── PAMR ──────────────────────────────────────────────────────────────────
import sys as _sys
_sys.path.insert(0, '.')
from pamr import PAMR
pamr_module = PAMR(num_iter=PAMR_ITER, dilations=PAMR_DILATIONS).to(DEVICE).eval()
print(f'PAMR ready (iter={PAMR_ITER}, dilations={PAMR_DILATIONS})', flush=True)

# ── soft prompts ───────────────────────────────────────────────────────────
sp_candidates = list(Path('/kaggle/input').rglob('soft_prompts.pt'))
if not sp_candidates:
    sp_candidates = [p for p in Path('/kaggle/input').rglob('*.pt') if 'sam3' not in p.name]
if sp_candidates:
    soft_prompts = torch.load(str(sp_candidates[0]), weights_only=True).to(DEVICE)
    print(f'soft_prompts: {soft_prompts.shape}', flush=True)
else:
    soft_prompts = None
    print('WARNING: soft_prompts.pt not found — method D skipped', flush=True)

# ── model ──────────────────────────────────────────────────────────────────
from config_local import SAM3_CHECKPOINT
from sam3 import build_sam3_image_model
from sam3.model.sam3_image_processor import Sam3Processor

print('Loading SAM3...', flush=True)
model = build_sam3_image_model(
    bpe_path='./sam3/assets/bpe_simple_vocab_16e6.txt.gz',
    checkpoint_path=SAM3_CHECKPOINT, device=DEVICE)
model.eval()
for p in model.parameters(): p.requires_grad = False
print(f'GPU: {torch.cuda.get_device_name(0)}', flush=True)
processor = Sam3Processor(model, confidence_threshold=CONFIDENCE_THRESHOLD, device=DEVICE)

# ── prompt parsing ─────────────────────────────────────────────────────────
def parse_cls_file(path):
    words, indices = [], []
    for cls_idx, line in enumerate(Path(path).read_text().splitlines()):
        line = line.strip()
        if not line: continue
        for syn in line.split(','):
            syn = syn.strip()
            if syn:
                words.append(syn); indices.append(cls_idx)
    return words, indices

# Potsdam: 5 classes queried, clutter=BG fallback
P_SINGLE_WORDS   = ['impervious surface', 'building', 'low vegetation', 'tree', 'car']
P_SINGLE_IDX     = list(range(5))
P_MULTI_WORDS, P_MULTI_IDX = parse_cls_file('./configs/cls_potsdam.txt')
P_CLS_NAMES      = ['impervious', 'building', 'low_veg', 'tree', 'car', 'clutter']
P_EVAL_CLASSES   = [0, 1, 2, 3, 4]   # excl. clutter

# Hessen: all 6 classes queried (no clutter), railway is class 5
H_MULTI_WORDS, H_MULTI_IDX = parse_cls_file('./configs/cls_hessen.txt')
H_CLS_NAMES      = ['impervious', 'building', 'tree', 'car', 'low_veg', 'railway']
H_PALETTE        = np.array([[255,255,255],[0,0,255],[0,200,0],[255,255,0],[0,255,180],[180,0,180]], dtype=np.uint8)
H_SINGLE_WORDS   = ['impervious surface', 'building', 'tree', 'car', 'low vegetation', 'railway track']
H_SINGLE_IDX     = list(range(6))

print(f'Potsdam queries: {len(P_SINGLE_WORDS)} single | {len(P_MULTI_WORDS)} multi', flush=True)
print(f'Hessen queries:  {len(H_SINGLE_WORDS)} single | {len(H_MULTI_WORDS)} multi', flush=True)

# ── text embedding cache ───────────────────────────────────────────────────
def cache_text(words, with_soft=False):
    cache = []
    with torch.no_grad():
        for word in words:
            te = model.backbone.forward_text([word], device=DEVICE)
            if with_soft and soft_prompts is not None:
                lang_f = te['language_features']
                soft_m = torch.zeros(1, soft_prompts.shape[0], device=DEVICE,
                                     dtype=te['language_mask'].dtype)
                te['language_features'] = torch.cat([lang_f, soft_prompts.to(dtype=lang_f.dtype)], dim=0)
                te['language_mask']     = torch.cat([te['language_mask'], soft_m], dim=1)
            cache.append({k: v.cpu() for k, v in te.items()})
    return cache

print('Caching text embeddings...', flush=True)
# Potsdam caches
p_cache_single = cache_text(P_SINGLE_WORDS, with_soft=False)
p_cache_multi  = cache_text(P_MULTI_WORDS,  with_soft=False)
p_cache_ptsam  = cache_text(P_MULTI_WORDS,  with_soft=True) if soft_prompts is not None else None
# Hessen caches
h_cache_single = cache_text(H_SINGLE_WORDS, with_soft=False)
h_cache_multi  = cache_text(H_MULTI_WORDS,  with_soft=False)
h_cache_ptsam  = cache_text(H_MULTI_WORDS,  with_soft=True) if soft_prompts is not None else None
print('Done.', flush=True)

# ── helpers ────────────────────────────────────────────────────────────────
P_PALETTE = np.array([[255,255,255],[0,0,255],[0,255,255],[0,255,0],[255,255,0],[255,0,0]], dtype=np.uint8)
P_RGB_TO_IDX = {
    (255,255,255): 0, (0,0,255): 1, (0,255,255): 2,
    (0,255,0): 3,     (255,255,0): 4, (255,0,0): 5,
}

def rgb_to_gt_potsdam(path):
    arr = tifffile.imread(str(path))
    if arr.ndim == 2:
        gt = torch.from_numpy(arr.astype(np.int64))
        if gt.max() > 5: gt = gt - 1
        gt[gt < 0] = IGNORE_IDX
        return gt
    h, w = arr.shape[:2]
    gt = torch.full((h, w), IGNORE_IDX, dtype=torch.int64)
    for (r, g, b), idx in P_RGB_TO_IDX.items():
        m = (arr[:,:,0]==r) & (arr[:,:,1]==g) & (arr[:,:,2]==b)
        gt[m] = idx
    return gt

def to_rgb(label_np, palette, n_cls):
    out = np.zeros((*label_np.shape, 3), dtype=np.uint8)
    for c in range(n_cls): out[label_np == c] = palette[c]
    out[label_np == IGNORE_IDX] = [80, 80, 80]
    return out

def collect_scores(state, h, w, te_cache, cls_idx_list, n_cls):
    logits = torch.zeros((n_cls, h, w), device=DEVICE)
    for te_cpu, cls_idx in zip(te_cache, cls_idx_list):
        processor.reset_all_prompts(state)
        for k, v in te_cpu.items(): state['backbone_out'][k] = v.to(DEVICE)
        state['geometric_prompt'] = model._get_dummy_prompt()
        processor._forward_grounding(state)
        scores = torch.zeros((h, w), device=DEVICE)
        if state.get('masks_logits') is not None and state['masks_logits'].shape[0] > 0:
            for i in range(state['masks_logits'].shape[0]):
                il = state['masks_logits'][i].squeeze()
                if il.shape != (h, w):
                    il = F.interpolate(il.view(1,1,*il.shape), size=(h,w),
                                       mode='bilinear', align_corners=False).squeeze()
                scores = torch.max(scores, il * state['object_score'][i])
        sem = state['semantic_mask_logits'].squeeze()
        if sem.shape != (h, w):
            sem = F.interpolate(sem.view(1,1,*sem.shape), size=(h,w),
                                mode='bilinear', align_corners=False).squeeze()
        scores = torch.max(scores, sem) * state['presence_score']
        logits[cls_idx] = torch.max(logits[cls_idx], scores)
    return logits

def pamr_crop(logits, crop_rgb):
    img_t = torch.from_numpy(crop_rgb.astype(np.float32)/255.).permute(2,0,1).unsqueeze(0).to(DEVICE)
    with torch.no_grad():
        return pamr_module(img_t, logits.unsqueeze(0)).squeeze(0)

def make_gauss(h, w):
    sy, sx = h/4., w/4.
    y = torch.arange(h, device=DEVICE).float() - (h-1)/2.
    x = torch.arange(w, device=DEVICE).float() - (w-1)/2.
    return torch.exp(-y[:,None]**2/(2*sy**2)) * torch.exp(-x[None,:]**2/(2*sx**2))

def finalize(acc, wt, bg_idx, prob_thd=PROB_THD):
    p = acc / wt.unsqueeze(0)
    seg = p.argmax(0)
    seg[p.max(0)[0] < prob_thd] = bg_idx
    return seg.cpu()

def compute_miou(seg_pred, gt, eval_cls, n_cls):
    tp = torch.zeros(n_cls); fp = torch.zeros(n_cls); fn = torch.zeros(n_cls)
    valid = (gt != IGNORE_IDX)
    for c in range(n_cls):
        pc = (seg_pred==c) & valid; lc = (gt==c) & valid
        tp[c]=(pc&lc).sum().float(); fp[c]=(pc&~lc).sum().float(); fn[c]=(~pc&lc).sum().float()
    iou = tp/(tp+fp+fn+1e-6)
    return iou, iou[eval_cls].mean().item()*100

def slide_infer(img_arr, n_cls, cache_single, idx_single, cache_multi, idx_multi,
                cache_ptsam, idx_ptsam, bg_idx, stride=STRIDE):
    H, W = img_arr.shape[:2]
    h_grids = max(H-CROP_SIZE+stride-1,0)//stride+1
    w_grids = max(W-CROP_SIZE+stride-1,0)//stride+1
    total = h_grids*w_grids
    gauss = make_gauss(CROP_SIZE, CROP_SIZE)
    acc_A = torch.zeros(n_cls,H,W,device=DEVICE)
    acc_B = torch.zeros(n_cls,H,W,device=DEVICE)
    acc_C = torch.zeros(n_cls,H,W,device=DEVICE)
    acc_D = torch.zeros(n_cls,H,W,device=DEVICE) if cache_ptsam else None
    wt    = torch.zeros(H,W,device=DEVICE)
    for hi in range(h_grids):
        for wi in range(w_grids):
            y1=hi*stride; x1=wi*stride
            y2=min(y1+CROP_SIZE,H); x2=min(x1+CROP_SIZE,W)
            y1=max(y2-CROP_SIZE,0); x1=max(x2-CROP_SIZE,0)
            crop_rgb = img_arr[y1:y2,x1:x2]
            h_c,w_c = y2-y1,x2-x1
            with torch.no_grad(), torch.autocast('cuda', dtype=torch.bfloat16):
                state = processor.set_image(Image.fromarray(crop_rgb))
                lA = collect_scores(state,h_c,w_c,cache_single,idx_single,n_cls).float()
                lB = collect_scores(state,h_c,w_c,cache_multi, idx_multi, n_cls).float()
                lD = collect_scores(state,h_c,w_c,cache_ptsam, idx_ptsam, n_cls).float() if cache_ptsam else None
            lC = pamr_crop(lB, crop_rgb)
            lD_pamr = pamr_crop(lD, crop_rgb) if lD is not None else None
            g = gauss[:h_c,:w_c]
            acc_A[:,y1:y2,x1:x2] += lA*g.unsqueeze(0)
            acc_B[:,y1:y2,x1:x2] += lB*g.unsqueeze(0)
            acc_C[:,y1:y2,x1:x2] += lC*g.unsqueeze(0)
            if acc_D is not None: acc_D[:,y1:y2,x1:x2] += lD_pamr*g.unsqueeze(0)
            wt[y1:y2,x1:x2] += g
    done = hi*w_grids+wi+1
    return (
        finalize(acc_A,wt,bg_idx), finalize(acc_B,wt,bg_idx),
        finalize(acc_C,wt,bg_idx),
        finalize(acc_D,wt,bg_idx) if acc_D is not None else None,
        total
    )

def infer_single_crop(crop_pil, n_cls, cache_single, idx_single,
                      cache_multi, idx_multi, cache_ptsam, idx_ptsam,
                      bg_idx, crop_rgb):
    h,w = crop_pil.size[1], crop_pil.size[0]
    with torch.no_grad(), torch.autocast('cuda', dtype=torch.bfloat16):
        state = processor.set_image(crop_pil)
        lA = collect_scores(state,h,w,cache_single,idx_single,n_cls).float()
        lB = collect_scores(state,h,w,cache_multi, idx_multi, n_cls).float()
        lD = collect_scores(state,h,w,cache_ptsam, idx_ptsam, n_cls).float() if cache_ptsam else None
    lC = pamr_crop(lB, crop_rgb)
    lD_pamr = pamr_crop(lD, crop_rgb) if lD is not None else None
    dummy_wt = torch.ones(h, w, device=DEVICE)
    seg_A = finalize(lA, dummy_wt, bg_idx, prob_thd=0.0)
    seg_B = finalize(lB, dummy_wt, bg_idx, prob_thd=0.0)
    seg_C = finalize(lC, dummy_wt, bg_idx, prob_thd=0.0)
    seg_D = finalize(lD_pamr, dummy_wt, bg_idx, prob_thd=0.0) if lD_pamr is not None else None
    return seg_A, seg_B, seg_C, seg_D

OUT = Path('/kaggle/working')
SCALE = 1500
def thumb(arr, maxdim=SCALE):
    h,w = arr.shape[:2]
    s = maxdim/max(h,w)
    return np.array(Image.fromarray(arr).resize((int(w*s),int(h*s)), Image.NEAREST))

# ═══════════════════════════════════════════════════════════════════════════
# PART 1 — POTSDAM MULTI-TILE
# ═══════════════════════════════════════════════════════════════════════════
print('\n' + '='*60)
print('PART 1 — POTSDAM MULTI-TILE ABLATION')
print('='*60, flush=True)

p_results = {}  # tile -> {method: (iou, miou)}
METHOD_NAMES = ['A  ZS-single', 'B  ZS-multi', 'C  ZS+PAMR', 'D  PTSAM+PAMR']

for tile in POTSDAM_TILES:
    print(f'\n--- Tile {tile} ---', flush=True)
    img_path = next(POTSDAM_ROOT.rglob(f'*{tile}*_RGB.tif'), None)
    lbl_path = next(POTSDAM_ROOT.rglob(f'*{tile}*_label_noBoundary.tif'), None)
    if img_path is None:
        print(f'  SKIP: tile {tile} not found'); continue
    img_arr = tifffile.imread(str(img_path))[:,:,:3]
    gt_full  = rgb_to_gt_potsdam(lbl_path)
    H,W = img_arr.shape[:2]
    print(f'  {W}x{H}  valid GT: {(gt_full!=IGNORE_IDX).sum():,}', flush=True)

    segA, segB, segC, segD, n_crops = slide_infer(
        img_arr, N_CLASSES_P,
        p_cache_single, P_SINGLE_IDX,
        p_cache_multi,  P_MULTI_IDX,
        p_cache_ptsam,  P_MULTI_IDX,
        BG_IDX_P)
    print(f'  {n_crops} crops done', flush=True)

    segs = [segA, segB, segC, segD]
    tile_res = {}
    print(f'  {"method":<22}', end='')
    for n in P_CLS_NAMES[:5]: print(f'{n:>11}', end='')
    print(f'  {"mIoU":>8}')
    for name, seg in zip(METHOD_NAMES, segs):
        if seg is None:
            print(f'  {name:<22}  (skipped)'); continue
        iou, miou = compute_miou(seg, gt_full, P_EVAL_CLASSES, N_CLASSES_P)
        tile_res[name] = (iou, miou)
        row = f'  {name:<22}'
        for c in P_EVAL_CLASSES: row += f'{iou[c]*100:10.2f}%'
        print(row + f'  {miou:7.2f}%')
    p_results[tile] = tile_res

    # visualize: 4 crop rows (full + 3 crops), 6 cols (RGB, GT, A, B, C, D)
    gt_rgb = to_rgb(gt_full.numpy(), P_PALETTE, N_CLASSES_P)
    seg_rgbs = [to_rgb(s.numpy(), P_PALETTE, N_CLASSES_P) if s is not None else gt_rgb for s in segs]
    crops = [
        ('top-left',     0,         0),
        ('centre',       H//2-512,  W//2-512),
        ('bottom-right', H-1024,    W-1024),
    ]
    fig, axes = plt.subplots(4, 6, figsize=(36, 24),
                             gridspec_kw={'hspace':0.05,'wspace':0.02})
    def get_m(name): return f"{tile_res[name][1]:.1f}%" if name in tile_res else 'n/a'
    titles = ['RGB','GT (excl. boundary)',
              f'A ZS-single\n{get_m("A  ZS-single")}',
              f'B ZS-multi\n{get_m("B  ZS-multi")}',
              f'C ZS+PAMR\n{get_m("C  ZS+PAMR")}',
              f'D PTSAM+PAMR\n{get_m("D  PTSAM+PAMR")}']
    for j,t in enumerate(titles): axes[0,j].set_title(t, fontsize=11, fontweight='bold')
    for j,im in enumerate([thumb(img_arr), thumb(gt_rgb)] + [thumb(s) for s in seg_rgbs]):
        axes[0,j].imshow(im); axes[0,j].axis('off')
    axes[0,0].set_ylabel('full tile', fontsize=11)
    for row,(lbl,y1c,x1c) in enumerate(crops, start=1):
        y1c=max(0,min(y1c,H-1024)); x1c=max(0,min(x1c,W-1024))
        y2c,x2c=y1c+1024,x1c+1024
        ims=[img_arr[y1c:y2c,x1c:x2c], gt_rgb[y1c:y2c,x1c:x2c]]+[s[y1c:y2c,x1c:x2c] for s in seg_rgbs]
        for j,im in enumerate(ims): axes[row,j].imshow(im); axes[row,j].axis('off')
        axes[row,0].set_ylabel(lbl, fontsize=11)
    patches=[mpatches.Patch(color=P_PALETTE[i]/255.,label=P_CLS_NAMES[i]) for i in range(N_CLASSES_P)]
    fig.legend(handles=patches, loc='lower center', ncol=N_CLASSES_P,
               fontsize=11, frameon=True, bbox_to_anchor=(0.5,0.01))
    summary='  |  '.join(f"{n.split()[0]}: {tile_res[n][1]:.1f}%" for n in METHOD_NAMES if n in tile_res)
    val_note = ' ← official val' if tile=='6_15' else ' (train tile, visual only)'
    fig.suptitle(f'Tile {tile}{val_note}  |  lit=57.8% (5-cls)\n{summary}',
                 fontsize=13, fontweight='bold', y=1.001)
    fig.savefig(str(OUT/f'potsdam_{tile}.png'), dpi=150, bbox_inches='tight')
    plt.close(fig)
    print(f'  Saved potsdam_{tile}.png', flush=True)

# summary table across tiles
print('\n=== Potsdam summary (mIoU 5-class) ===')
print(f'  {"tile":<8}', end='')
for n in METHOD_NAMES: print(f'{n:>16}', end='')
print()
for tile,tres in p_results.items():
    print(f'  {tile:<8}', end='')
    for n in METHOD_NAMES:
        if n in tres: print(f'{tres[n][1]:15.2f}%', end='')
        else:         print(f'{"n/a":>16}', end='')
    print()
print(f'  Literature zero-shot: 57.8%')

# ═══════════════════════════════════════════════════════════════════════════
# PART 2 — DARMSTADT
# ═══════════════════════════════════════════════════════════════════════════
print('\n' + '='*60)
print('PART 2 — DARMSTADT DOP20 (visual only, cls_hessen.txt)')
print('='*60, flush=True)

all_patches = sorted(DARMSTADT_ROOT.glob('*.png'))
print(f'Found {len(all_patches)} Darmstadt patches', flush=True)

# ── 2a: 5 random patches ───────────────────────────────────────────────────
sampled = random.sample(all_patches, min(N_DARMSTADT_PATCHES, len(all_patches)))
print(f'\n--- 2a: {len(sampled)} random patches ---', flush=True)

fig, axes = plt.subplots(len(sampled), 6,
                         figsize=(36, 6*len(sampled)),
                         gridspec_kw={'hspace':0.08,'wspace':0.02})
col_titles = ['RGB', 'A ZS-single', 'B ZS-multi', 'C ZS+PAMR',
              'D PTSAM+PAMR\n(cross-domain)', '']
for j,t in enumerate(col_titles[:5]): axes[0,j].set_title(t, fontsize=11, fontweight='bold')
axes[0,5].axis('off')

for row, patch_path in enumerate(sampled):
    crop_rgb = np.array(Image.open(patch_path).convert('RGB'))
    crop_pil = Image.fromarray(crop_rgb)
    sA,sB,sC,sD = infer_single_crop(
        crop_pil, N_CLASSES_H,
        h_cache_single, H_SINGLE_IDX,
        h_cache_multi,  H_MULTI_IDX,
        h_cache_ptsam,  H_MULTI_IDX,
        BG_IDX_H, crop_rgb)
    segs_h = [sA, sB, sC, sD]
    ims = [crop_rgb] + [to_rgb(s.numpy(), H_PALETTE, N_CLASSES_H) if s is not None else crop_rgb for s in segs_h]
    for j,im in enumerate(ims[:5]):
        axes[row,j].imshow(im); axes[row,j].axis('off')
    axes[row,5].axis('off')
    axes[row,0].set_ylabel(patch_path.name[:30], fontsize=8)
    print(f'  {patch_path.name}', flush=True)

h_patches=[mpatches.Patch(color=H_PALETTE[i]/255.,label=H_CLS_NAMES[i]) for i in range(N_CLASSES_H)]
fig.legend(handles=h_patches, loc='lower center', ncol=N_CLASSES_H,
           fontsize=11, frameon=True, bbox_to_anchor=(0.5,0.01))
fig.suptitle('Darmstadt DOP20 — 5 random patches — 4 methods (cls_hessen)',
             fontsize=13, fontweight='bold', y=1.001)
fig.savefig(str(OUT/'darmstadt_patches.png'), dpi=150, bbox_inches='tight')
plt.close(fig)
print('  Saved darmstadt_patches.png', flush=True)

# ── 2b: full reconstructed tile ────────────────────────────────────────────
print('\n--- 2b: full Darmstadt tile (stitched) ---', flush=True)

import re
def parse_patch(p):
    m = re.search(r'y(\d+)_x(\d+)', p.name)
    return (int(m.group(1)), int(m.group(2))) if m else None

# group patches by tile base (everything before _yN_xN)
tile_groups = {}
for p in all_patches:
    base = re.sub(r'_y\d+_x\d+.*', '', p.stem)
    tile_groups.setdefault(base, []).append(p)
best_base = max(tile_groups, key=lambda b: len(tile_groups[b]))
tile_patches = sorted(tile_groups[best_base], key=lambda p: parse_patch(p))
print(f'  Tile base: {best_base}  ({len(tile_patches)} patches)', flush=True)

# determine grid size
coords = [parse_patch(p) for p in tile_patches]
max_y  = max(c[0] for c in coords)
max_x  = max(c[1] for c in coords)
patch_h, patch_w = np.array(Image.open(tile_patches[0])).shape[:2]
full_h = (max_y + 1) * patch_h
full_w = (max_x + 1) * patch_w
print(f'  Grid: {max_y+1}x{max_x+1} patches -> {full_w}x{full_h} image', flush=True)

full_img = np.zeros((full_h, full_w, 3), dtype=np.uint8)
coord_map = {parse_patch(p): p for p in tile_patches}
for (y,x), p in coord_map.items():
    patch = np.array(Image.open(p).convert('RGB'))
    y0,x0 = y*patch_h, x*patch_w
    full_img[y0:y0+patch_h, x0:x0+patch_w] = patch
print(f'  Stitched full image: {full_img.shape}', flush=True)

# run sliding window on full stitched image
print('  Running sliding window...', flush=True)
sA_f, sB_f, sC_f, sD_f, nc = slide_infer(
    full_img, N_CLASSES_H,
    h_cache_single, H_SINGLE_IDX,
    h_cache_multi,  H_MULTI_IDX,
    h_cache_ptsam,  H_MULTI_IDX,
    BG_IDX_H, stride=patch_h)  # stride=256 = patch size
print(f'  {nc} crops done', flush=True)

segs_full = [sA_f, sB_f, sC_f, sD_f]
titles_full = ['RGB', 'A ZS-single', 'B ZS-multi', 'C ZS+PAMR', 'D PTSAM+PAMR']
fig, axes = plt.subplots(1, 5, figsize=(40, 10),
                         gridspec_kw={'hspace':0.02,'wspace':0.02})
for j,t in enumerate(titles_full): axes[j].set_title(t, fontsize=12, fontweight='bold')
ims_full = [thumb(full_img)] + [
    thumb(to_rgb(s.numpy(), H_PALETTE, N_CLASSES_H)) if s is not None else thumb(full_img)
    for s in segs_full]
for j,im in enumerate(ims_full): axes[j].imshow(im); axes[j].axis('off')

fig.legend(handles=h_patches, loc='lower center', ncol=N_CLASSES_H,
           fontsize=11, frameon=True, bbox_to_anchor=(0.5,-0.02))
fig.suptitle(f'Darmstadt full tile: {best_base}  ({full_w}x{full_h})\ncls_hessen — 4 methods',
             fontsize=13, fontweight='bold')
fig.savefig(str(OUT/'darmstadt_full.png'), dpi=150, bbox_inches='tight')
plt.close(fig)
print('  Saved darmstadt_full.png', flush=True)

print('\nAll outputs saved to /kaggle/working/', flush=True)
print('  potsdam_6_15.png  potsdam_5_14.png  potsdam_7_13.png', flush=True)
print('  darmstadt_patches.png  darmstadt_full.png', flush=True)
PYEOF